In [1]:
from unsloth import FastLanguageModel
from transformers import TextStreamer, AutoTokenizer
from peft import PeftModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# ── Config ─────────────────────────────────────────────────────────────────────
base_model_name = "/home/jovyan/work/sinllama/models/llama-3-8b"
adapter_name    = "/home/jovyan/work/sinllama/models/SinLlama_v01"
tokenizer_name  = "/home/jovyan/work/sinllama/models/Extended-Sinhala-LLaMA"

max_seq_length  = 2048
dtype           = torch.bfloat16
load_in_4bit    = True   # 🔥 IMPORTANT (prevents OOM)

In [3]:
# ── Load tokenizer ─────────────────────────────────────────────────────────────
print("🔹 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_name,
    local_files_only=True,
)

🔹 Loading tokenizer...


In [4]:
# ── Load base model ────────────────────────────────────────────────────────────
print("🔹 Loading base model...")
model, _ = FastLanguageModel.from_pretrained(
    model_name       = base_model_name,
    max_seq_length   = max_seq_length,
    dtype            = dtype,
    load_in_4bit     = load_in_4bit,
    local_files_only = True,
)

🔹 Loading base model...
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.352 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /home/jovyan/work/sinllama/models/llama-3-8b as a legacy tokenizer.


In [5]:
# ── 🔥 SAFE EMBEDDING RESIZE (CPU to avoid OOM) ────────────────────────────────
print("🔹 Resizing embeddings safely (CPU)...")
model = model.to("cpu")
model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
model = model.to("cuda")

🔹 Resizing embeddings safely (CPU)...


In [6]:
# ── Load LoRA adapter ──────────────────────────────────────────────────────────
print("🔹 Loading SinLlama adapter...")
model = PeftModel.from_pretrained(
    model,
    adapter_name,
    local_files_only=True,
    ensure_weight_tying=True,
)

🔹 Loading SinLlama adapter...


In [7]:
# ── Enable fast inference ──────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): ModulesToSaveWrapper(
          (original_module): Embedding(139336, 4096, padding_idx=128255)
          (modules_to_save): ModuleDict(
            (default): Embedding(139336, 4096, padding_idx=128255)
          )
        )
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )


In [8]:
# ── Prompt formatter (STRONG CONTROL) ──────────────────────────────────────────
def format_prompt(user_input: str):
    return f"""### Instruction:
You are a helpful Sinhala AI assistant.
Answer ONLY the question in 1–2 short sentences.
Do NOT generate stories or extra text.

Question: {user_input}

### Response:
"""

# ── Generation config ──────────────────────────────────────────────────────────
def get_generation_config():
    return {
        "max_new_tokens": 100,
        "temperature": 0.2,
        "top_p": 0.7,
        "top_k": 40,
        "repetition_penalty": 1.2,
        "do_sample": True,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id,
    }

In [ ]:
# ── Inference loop ─────────────────────────────────────────────────────────────
def run():
    print("\n✅ SinLlama ready!")
    print("   Sinhala input gives best results.")
    print("   Type 'quit' to exit.\n")

    gen_config = get_generation_config()
    streamer   = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    while True:
        prompt = input("\n📝 Prompt: ").strip()

        if not prompt:
            continue
        if prompt.lower() == "quit":
            print("👋 Bye!")
            break

        formatted_prompt = format_prompt(prompt)

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

        print("\n🤖 Output:\n")

        with torch.no_grad():
            model.generate(
                **inputs,
                streamer=streamer,
                **gen_config,
            )

# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run()


✅ SinLlama ready!
   Sinhala input gives best results.
   Type 'quit' to exit.




📝 Prompt:  කොහොමද?



🤖 Output:



Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.11/

aaතර”	"aageout‍,“කු-ieourන් u ourac’‍ය " “adazෂ. otheragටuක itගෝ‍ලේ​නිෂ්000සැ202	P (abahav\ �	Mapooතීib_‘ද oniඉ at’’අ) Post S the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the


...

Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤖 Output:

aa”	" "“a,‍-ageoutතරacadagu “කකු‍ලේ202​.’azabie‍ය S000 (	Pibourනි ourapට it u	M � onooගෝ The Post post -‍ට at�ata­ Agah201දdඅ\ka වල B‘”) එක Aඅප the the the the the the the the the the the the the the the the the the the the the the the the the the the the



📝 Prompt:  How are you?


Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤖 Output:

aaතර”"	“‍auageout u,adagac-න්ourකු’ieගෝazසැ​කට “ ". ourap‍ලේ (abනි‍ය S other000202	P oniඅ �…oo it­ib onlined\ The)ahang	M�\tෂ - the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the ro the the the
